In [1]:
import cv2
import numpy as np
import time
from datetime import datetime
import csv
import os
import pickle
from sklearn.neighbors import KNeighborsClassifier

# ── YOLOv8 via ultralytics ────────────────────────────────────────────
from ultralytics import YOLO   # pip install ultralytics


class DeskTracker:
    def __init__(self):
        # ── YOLOv8 model ─────────────────────────────────────────────
        # Replace 'yolov8n.pt' with any YOLOv8 variant:
        #   yolov8n.pt  (nano  – fastest / smallest)
        #   yolov8s.pt  (small)
        #   yolov8m.pt  (medium)
        #   yolov8l.pt  (large)
        #   yolov8x.pt  (extra-large – most accurate)
        # The first run downloads the weights automatically.
        self.model = YOLO('Yolo_model/yolov8n.pt')

        # COCO class indices used in desk-area detection
        # 56=chair, 60=dining table, 63=laptop, 64=mouse, 65=remote
        self.DESK_RELATED_CLASSES = {56, 60, 63, 64, 65}
        self.PERSON_CLASS_ID = 0           # COCO class 0 = person

        # Detection thresholds
        self.confidence_threshold = 0.3    # used during desk-area scan
        self.person_conf = 0.50            # used during tracking

        # ── Face recognition ─────────────────────────────────────────
        self.face_recognition_threshold = 18
        self.face_min_size = (30, 30)

        try:
            with open('data/Organizations.pkl', 'rb') as f:
                self.ORGANIZATIONS = pickle.load(f)
            with open('data/names.pkl', 'rb') as f:
                self.NAMES = pickle.load(f)
            with open('data/Emails.pkl', 'rb') as f:
                self.EMAILS = pickle.load(f)
            with open('data/EmployeeIDs.pkl', 'rb') as f:
                self.EmployeeIDs = pickle.load(f)
            with open('data/faces_data.pkl', 'rb') as f:
                self.FACES = pickle.load(f)
                self.FACES = self.FACES / 255.0
                print(f"FACES shape: {self.FACES.shape}")

            self.knn = KNeighborsClassifier(n_neighbors=5)
            self.knn.fit(self.FACES, self.EmployeeIDs)
            print("Face recognition loaded successfully.")
            self.face_recognition_enabled = True
        except Exception as e:
            print(f"Face model error: {e}. Using 'Unknown' for all.")
            self.NAMES = []
            self.EmployeeIDs = []
            self.ORGANIZATIONS = []
            self.EMAILS = []
            self.face_recognition_enabled = False

        self.face_cascade = cv2.CascadeClassifier(
            'data/haarcascade_frontalface_default.xml')

        # ── Desk ROI ─────────────────────────────────────────────────
        self.DESK_ROI_PT1 = None
        self.DESK_ROI_PT2 = None
        self.desk_mask = None

        # ── Tracking state ───────────────────────────────────────────
        self.trackers = {}
        self.person_status = {}
        self.MAX_CENTROIDS = 10
        self.DISAPPEARANCE_TIME = 2.0

        self.employee_absence_start = None
        self.visitor_absence_start = None
        self.last_employee_present = None
        self.last_visitor_present = None

        # ── Logging setup ────────────────────────────────────────────
        date = datetime.now().strftime("%d-%m-%Y")
        os.makedirs('Logs', exist_ok=True)
        log_folder = os.path.join('Logs', f'Log_{date}')
        os.makedirs(log_folder, exist_ok=True)

        self.log_file = os.path.join(log_folder, f'detection_log_{date}.csv')
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w', newline='') as f:
                csv.writer(f).writerow(
                    ['Date & Time', 'EmployeeID', 'Name', 'Detection Status'])

        self.absence_log_file = os.path.join(
            log_folder, f'absence_log_{date}.csv')
        if not os.path.exists(self.absence_log_file):
            with open(self.absence_log_file, 'w', newline='') as f:
                csv.writer(f).writerow(
                    ['Date & Time', 'EmployeeID', 'Name', 'Absence Duration (s)'])

        self.person_absence_start = {}
        self.person_absence_logged = {}

        self.monitor_area = None
        self.display_width = 1280
        self.display_height = 720
        self.debug_mode = False

    # ─────────────────────────────────────────────────────────────────
    # LOGGING HELPERS
    # ─────────────────────────────────────────────────────────────────
    def log_event(self, message):
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}")

    def log_detection(self, name, employee_id, status):
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.log_file, 'a', newline='') as f:
            csv.writer(f).writerow([timestamp, employee_id, name, status])
        self.log_event(f"[DETECTION] {name} ({employee_id}) → {status}")

    def log_absence(self, name, employee_id, duration_seconds):
        if duration_seconds <= 3:
            return
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        with open(self.absence_log_file, 'a', newline='') as f:
            csv.writer(f).writerow(
                [timestamp, employee_id, name, f"{duration_seconds:.1f}"])
        self.log_event(
            f"[ABSENCE]   {name} ({employee_id}) absent for {duration_seconds:.1f}s")

    # ─────────────────────────────────────────────────────────────────
    # PER-PERSON ABSENCE TRACKING
    # ─────────────────────────────────────────────────────────────────
    def mark_person_present(self, name, employee_id):
        if name in self.person_absence_start:
            duration = time.time() - self.person_absence_start.pop(name)
            if name != "People":
                self.log_absence(name, employee_id, duration)

    def mark_person_absent(self, name):
        if name not in self.person_absence_start:
            self.person_absence_start[name] = time.time()

    # ─────────────────────────────────────────────────────────────────
    # DESK AREA DETECTION  (YOLOv8 version)
    # ─────────────────────────────────────────────────────────────────
    def detect_desk_area(self, cap):
        self.log_event("Detecting desk area with YOLOv8 …")
        desk_candidates = []
        frame_count = 0
        frame = None

        while frame_count < 5:
            ret, frame = cap.read()
            if not ret or frame is None or frame.size == 0:
                time.sleep(0.05)
                continue

            height, width = frame.shape[:2]

            # ── YOLOv8 inference ─────────────────────────────────────
            results = self.model(
                frame,
                conf=self.confidence_threshold,
                verbose=False
            )

            for box in results[0].boxes:
                cls_id = int(box.cls[0])
                conf   = float(box.conf[0])
                if cls_id in self.DESK_RELATED_CLASSES:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    desk_candidates.append(
                        [x1, y1, x2 - x1, y2 - y1, conf, cls_id])

            frame_count += 1

        # Compute ROI from candidates
        if len(desk_candidates) < 2:
            self.log_event(
                "Not enough desk objects detected – using default area.")
            if frame is not None:
                height, width = frame.shape[:2]               
            else:
                height, width = 480, 640
            self.log_event(
                "Now Using Default area with (height, width = 480, 640).") 
            min_x = int(width  * 0.2)
            min_y = int(height * 0.4)
            max_x = int(width  * 0.8)
            max_y = int(height * 0.9)
        else:
            x_coords     = [d[0]          for d in desk_candidates]
            y_coords     = [d[1]          for d in desk_candidates]
            max_x_coords = [d[0] + d[2]   for d in desk_candidates]
            max_y_coords = [d[1] + d[3]   for d in desk_candidates]
            min_x = max(0,      int(min(x_coords)     - 0.1 * width))
            min_y = max(0,      int(min(y_coords)     - 0.1 * height))
            max_x = min(width,  int(max(max_x_coords) + 0.1 * width))
            max_y = min(height, int(max(max_y_coords) + 0.1 * height))
            self.log_event(
                f"Desk area detected: ({min_x}, {min_y}) → ({max_x}, {max_y})")

        self.DESK_ROI_PT1 = (min_x, min_y)
        self.DESK_ROI_PT2 = (max_x, max_y)
        self.monitor_area = (min_x, min_y, max_x, max_y)
        return (min_x, min_y, max_x, max_y)

    # ─────────────────────────────────────────────────────────────────
    # FACE RECOGNITION
    # ─────────────────────────────────────────────────────────────────
    def get_name_from_face(self, face_region):
        if not self.face_recognition_enabled or face_region.size == 0:
            return "People", "N/A", ""
        if face_region.shape[0] < 30 or face_region.shape[1] < 30:
            return "People", "N/A", ""

        gray_region = cv2.cvtColor(face_region, cv2.COLOR_BGR2GRAY)
        faces = []
        for scale, neighbors, min_size in [
            (1.1,  5, (20, 20)),
            (1.05, 4, (25, 25)),
            (1.2,  5, (30, 30)),
            (1.3,  4, (35, 35)),
            (1.15, 3, (25, 25)),
        ]:
            detected = self.face_cascade.detectMultiScale(
                gray_region, scale, neighbors, minSize=min_size)
            if len(detected) > 0:
                faces = detected
                break

        Name, EmployeeID, Email = "People", "N/A", ""

        for (x, y, w_face, h_face) in faces:
            crop_img = face_region[y:y+h_face, x:x+w_face, :]
            if crop_img.size == 0:
                continue
            try:
                resized_img  = cv2.resize(crop_img, (75, 100))
                gray         = cv2.cvtColor(resized_img, cv2.COLOR_BGR2GRAY)
                resized_flat = gray.reshape(1, -1) / 255.0
                if resized_flat.shape[1] != self.FACES.shape[1]:
                    continue
                distances, _ = self.knn.kneighbors(resized_flat)
                if distances[0][0] <= self.face_recognition_threshold:
                    output = self.knn.predict(resized_flat)
                    emp_id = output[0]
                    if emp_id in self.EmployeeIDs:
                        index      = self.EmployeeIDs.index(emp_id)
                        Name       = self.NAMES[index]
                        EmployeeID = self.EmployeeIDs[index] if index < len(self.EmployeeIDs) else "N/A"
                        Email      = self.EMAILS[index]      if index < len(self.EMAILS)      else ""
                        break
            except Exception:
                continue

        return Name, EmployeeID, Email

    def get_name_from_centroid(self, cx, cy, frame):
        h, w = frame.shape[:2]
        search_regions = [
            (max(0,cx-180), max(0,cy-280), min(w,cx+180), min(h,cy+50)),
            (max(0,cx-220), max(0,cy-320), min(w,cx+220), min(h,cy+80)),
            (max(0,cx-150), max(0,cy-350), min(w,cx+150), min(h,cy)),
            (max(0,cx-200), max(0,cy-200), min(w,cx+200), min(h,cy+150)),
        ]
        for idx, (rx1, ry1, rx2, ry2) in enumerate(search_regions):
            if self.debug_mode:
                color = [(255,0,255),(0,255,255),(255,255,0),(255,128,0)][idx]
                cv2.rectangle(frame, (rx1,ry1), (rx2,ry2), color, 2)
                cv2.putText(frame, f"Search {idx+1}", (rx1, ry1-5),
                            cv2.FONT_HERSHEY_COMPLEX, 0.4, color, 1)
            face_region = frame[ry1:ry2, rx1:rx2]
            name, EmployeeID, email = self.get_name_from_face(face_region)
            if name != "People":
                return name, EmployeeID, email
        return "People", "N/A", ""

    # ─────────────────────────────────────────────────────────────────
    # MAIN RUN
    # ─────────────────────────────────────────────────────────────────
    def run(self):
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            self.log_event("Error: Cannot open default camera.")
            return
        self.log_event("Default camera opened.")

        cap.set(cv2.CAP_PROP_FRAME_WIDTH,  1920)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)
        cap.set(cv2.CAP_PROP_FPS, 30)

        actual_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        actual_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.log_event(f"Resolution: {actual_width}x{actual_height}")

        self.detect_desk_area(cap)
        self.log_event("Starting tracker… Press 'q' to quit.")
        self._run_full_mode(cap)

    # ─────────────────────────────────────────────────────────────────
    # FULL MODE  (YOLOv8 version)
    # ─────────────────────────────────────────────────────────────────
    def _run_full_mode(self, cap):
        last_cleanup = time.time()

        cv2.namedWindow('Employee Desk Tracking', cv2.WINDOW_NORMAL)
        cv2.resizeWindow('Employee Desk Tracking',
                         self.display_width, self.display_height)

        while True:
            ret, frame = cap.read()
            if not ret or frame is None or frame.size == 0:
                time.sleep(0.05)
                continue

            h, w = frame.shape[:2]

            # Rebuild desk mask if needed
            if self.desk_mask is None or self.desk_mask.shape[:2] != (h, w):
                self.desk_mask = np.zeros((h, w), np.uint8)
                if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                    cv2.rectangle(self.desk_mask,
                                  self.DESK_ROI_PT1, self.DESK_ROI_PT2,
                                  255, -1)

            # ── YOLOv8 inference ─────────────────────────────────────
            results = self.model(
                frame,
                conf=self.person_conf,
                classes=[self.PERSON_CLASS_ID],   # person only
                verbose=False
            )

            current_in_desk   = {}
            current_outside   = {}
            employee_detected = False
            visitor_detected  = False
            seen_names        = set()

            for box in results[0].boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf            = float(box.conf[0])
                cx, cy          = (x1 + x2) // 2, (y1 + y2) // 2

                name, EmployeeID, email = self.get_name_from_centroid(
                    cx, cy, frame)
                seen_names.add(name)

                # Update tracker
                if name not in self.trackers:
                    self.trackers[name] = {
                        'centroids': [], 'last_seen': 0,
                        'EmployeeID': EmployeeID, 'email': email}
                self.trackers[name].update({
                    'bbox': (x1, y1, x2, y2),
                    'last_seen': time.time(),
                    'EmployeeID': EmployeeID,
                    'email': email})
                self.trackers[name]['centroids'].append((cx, cy))
                if len(self.trackers[name]['centroids']) > self.MAX_CENTROIDS:
                    self.trackers[name]['centroids'].pop(0)

                # Desk overlap
                overlap_ratio = 0
                if x2 > x1 and y2 > y1:
                    ys  = slice(max(0,y1), min(h,y2))
                    xs  = slice(max(0,x1), min(w,x2))
                    roi = self.desk_mask[ys, xs]
                    if roi.size > 0:
                        overlap_ratio = np.sum(roi > 0) / roi.size

                location = 'Inside Desk' if overlap_ratio > 0.4 else 'Outside Desk'

                if 'Inside' in location:
                    current_in_desk[name] = (EmployeeID, email)
                    if name != "People":
                        employee_detected = True
                        self.last_employee_present = time.time()
                        self.employee_absence_start = None
                else:
                    current_outside[name] = (EmployeeID, email)
                    visitor_detected = True
                    self.last_visitor_present = time.time()
                    self.visitor_absence_start = None

                # Log status changes (File 1)
                if name not in self.person_status:
                    self.person_status[name] = location
                    self.log_detection(name, EmployeeID,
                                       f"Detected - {location}")
                elif self.person_status[name] != location:
                    self.person_status[name] = location
                    self.log_detection(name, EmployeeID, location)

                # Mark present (File 2)
                self.mark_person_present(name, EmployeeID)

                # ── Draw bounding box ─────────────────────────────────
                color = (0, 255, 0) if 'Inside' in location else (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                role  = "Employee" if 'Inside' in location else "Visitor"
                label = f"{name} - {role} ({location.split()[0]})"
                lsz   = cv2.getTextSize(
                    label, cv2.FONT_HERSHEY_COMPLEX, 0.8, 2)[0]
                cv2.rectangle(frame,
                              (x1, y1-40), (x1+lsz[0]+15, y1), color, -1)
                cv2.putText(frame, label, (x1+8, y1-15),
                            cv2.FONT_HERSHEY_COMPLEX, 0.8, (255,255,255), 2)
                id_display = (EmployeeID
                              if EmployeeID and EmployeeID != "N/A"
                              else ("Employee" if 'Inside' in location
                                    else "Visitor"))
                cv2.putText(frame,
                            f"{id_display} | Conf: {conf:.2f}",
                            (x1, y2+25),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, color, 2)

            # Mark absent for people not seen this frame
            for tracked_name in list(self.person_status.keys()):
                if tracked_name not in seen_names:
                    self.mark_person_absent(tracked_name)

            if not employee_detected and self.employee_absence_start is None:
                self.employee_absence_start = time.time()
            if not visitor_detected and self.visitor_absence_start is None:
                self.visitor_absence_start = time.time()

            # Cleanup disappeared trackers
            if time.time() - last_cleanup > 1.0:
                to_remove = [
                    n for n, d in self.trackers.items()
                    if time.time() - d['last_seen'] > self.DISAPPEARANCE_TIME]
                for name in to_remove:
                    if name in self.person_status:
                        eid = self.trackers[name].get('EmployeeID', 'N/A')
                        self.log_detection(name, eid, "Disappeared")
                        del self.person_status[name]
                    del self.trackers[name]
                last_cleanup = time.time()

            # ── Draw desk ROI corners ─────────────────────────────────
            if self.DESK_ROI_PT1 and self.DESK_ROI_PT2:
                cv2.rectangle(frame,
                              self.DESK_ROI_PT1, self.DESK_ROI_PT2,
                              (255, 255, 0), 4)
                ms = 40
                p1, p2 = self.DESK_ROI_PT1, self.DESK_ROI_PT2
                for pt, dx, dy in [
                    (p1,            (ms, 0),   (0,  ms)),
                    ((p2[0], p1[1]),(-ms, 0),  (0,  ms)),
                    ((p1[0], p2[1]),(ms, 0),   (0, -ms)),
                    (p2,            (-ms, 0),  (0, -ms)),
                ]:
                    cv2.line(frame, pt,
                             (pt[0]+dx[0], pt[1]+dx[1]), (0,255,255), 6)
                    cv2.line(frame, pt,
                             (pt[0]+dy[0], pt[1]+dy[1]), (0,255,255), 6)

            # ── Status overlay ────────────────────────────────────────
            emp_text  = ("Employee Status: PRESENT"
                         if employee_detected else "Employee Status: ABSENT")
            emp_color = (0,255,0) if employee_detected else (0,0,255)
            cv2.putText(frame, emp_text, (15,35),
                        cv2.FONT_HERSHEY_COMPLEX, 0.7, emp_color, 2)

            if not employee_detected and self.employee_absence_start:
                dur = time.time() - self.employee_absence_start
                cv2.putText(frame, f"Employee Absence: {dur:.1f}s",
                            (15,65), cv2.FONT_HERSHEY_COMPLEX,
                            0.6, (0,0,255), 2)

            vis_count = len(current_outside)
            vis_text  = (f"Visitor Status: {vis_count} PRESENT"
                         if visitor_detected else "Visitor Status: NONE")
            vis_color = (0,165,255) if visitor_detected else (128,128,128)
            cv2.putText(frame, vis_text, (15,100),
                        cv2.FONT_HERSHEY_COMPLEX, 0.7, vis_color, 2)

            cv2.putText(
                frame,
                f"Inside: {len(current_in_desk)} | "
                f"Outside: {len(current_outside)} | "
                f"Total: {len(self.trackers)}",
                (15,135), cv2.FONT_HERSHEY_COMPLEX, 0.7, (255,255,0), 2)

            if current_in_desk:
                nin = ", ".join(list(current_in_desk.keys())[:3])
                if len(current_in_desk) > 3:
                    nin += f" +{len(current_in_desk)-3}"
                cv2.putText(frame, f"Inside Desk: {nin}", (15,165),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,255,0), 2)
            if current_outside:
                nout = ", ".join(list(current_outside.keys())[:3])
                if len(current_outside) > 3:
                    nout += f" +{len(current_outside)-3}"
                cv2.putText(frame, f"Outside Desk: {nout}", (15,195),
                            cv2.FONT_HERSHEY_COMPLEX, 0.6, (0,165,255), 2)

            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            cv2.putText(frame, f"Time: {ts}", (15,225),
                        cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,255,255), 2)
            cv2.putText(frame, "Press q to Quit", (15,250),
                        cv2.FONT_HERSHEY_COMPLEX, 0.6, (255,200,0), 2)

            cv2.imshow('Employee Desk Tracking', frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        cv2.destroyAllWindows()
        self.log_event(
            f"Logs saved → {self.log_file}  |  {self.absence_log_file}")


# ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import sys

    tracker = DeskTracker()

    for arg in sys.argv[1:]:
        if arg.lower() in ['debug', 'd', '--debug']:
            tracker.debug_mode = True
            print("DEBUG MODE ENABLED – face search regions visualised")

    print("\n" + "="*60)
    print("Starting Employee Desk Tracker  [YOLOv8]")
    print("="*60)
    print("\nPress 'q' to quit\n" + "="*60 + "\n")

    tracker.run()

FACES shape: (50, 7500)
Face recognition loaded successfully.

Starting Employee Desk Tracker  [YOLOv8]

Press 'q' to quit



[11:06:35] Default camera opened.


[11:06:38] Resolution: 1280x720
[11:06:38] Detecting desk area with YOLOv8 …


[11:06:39] Not enough desk objects detected – using default area.
[11:06:39] Now Using Default area with (height, width = 480, 640).
[11:06:39] Starting tracker… Press 'q' to quit.


[11:06:40] [DETECTION] People (N/A) → Detected - Inside Desk


...

[11:07:29] [DETECTION] Piyush (23bcs061) → Outside Desk


[11:07:45] [DETECTION] People (N/A) → Detected - Outside Desk


[11:07:46] [DETECTION] Piyush (23bcs061) → Disappeared


[11:07:47] [DETECTION] Piyush (23bcs061) → Detected - Outside Desk


[11:07:47] Logs saved → Logs\Log_18-02-2026\detection_log_18-02-2026.csv  |  Logs\Log_18-02-2026\absence_log_18-02-2026.csv
